# 🧪 AgentsVille AI Trip Planner – Test Scenarios

This notebook exercises the trip-planning system across **six scenarios** that
cover a range of traveler profiles, budget constraints, and weather challenges.

Each scenario follows the same structure:

1. Define `VacationInfo`
2. Gather weather & activity data
3. Generate an initial itinerary
4. Run all five evaluations
5. Revise with the ReAct agent until all checks pass
6. Inspect the reasoning log and key metrics

## 🔧 Setup

In [ ]:
import json
import os
import time
from pathlib import Path

from openai import OpenAI

from project_lib import (
    VacationInfo,
    TravelPlan,
    get_weather_forecast,
    get_available_activities,
    run_evals,
    ItineraryAgent,
    ItineraryRevisionAgent,
    generate_trip_summary,
    print_itinerary,
    print_eval_results,
)

client = OpenAI()   # reads OPENAI_API_KEY from environment

MAIN_MODEL  = "gpt-4o"
EVAL_MODEL  = "gpt-4o-mini"

print("✅ Setup complete")

---
## Scenario 1 – Budget-Conscious Travelers

* **Travelers:** 2 people, cultural interests
* **Budget:** $100 (tight constraint)
* **Duration:** 3 days (2026-06-10 → 2026-06-12)
* **Goal:** Verify the system stays within a very tight budget and selects affordable activities

In [ ]:
vacation_info_s1 = VacationInfo(
    destination="AgentsVille",
    start_date="2026-06-10",
    end_date="2026-06-12",
    interests=["culture", "history", "art"],
    budget=100.0,
    constraints=["low budget – prefer free or cheap activities"],
)

weather_s1     = get_weather_forecast(vacation_info_s1)
activities_s1  = get_available_activities(vacation_info_s1, weather_s1)

print("🌤️  Weather Forecast")
for d, w in sorted(weather_s1.items()):
    print(f"   {d}: {w}")

print()
print("🎯  Available Activities per Day")
for d in sorted(activities_s1):
    print(f"   {d} ({weather_s1[d]}): {len(activities_s1[d])} activities")

In [ ]:
t0 = time.time()

agent_s1         = ItineraryAgent(client=client, model=MAIN_MODEL)
initial_plan_s1  = agent_s1.generate(
    vacation_info=vacation_info_s1,
    weather_data=weather_s1,
    available_activities=activities_s1,
)

print("📋  Initial Itinerary – Scenario 1")
print_itinerary(initial_plan_s1)

In [ ]:
print("🔍  Initial Evaluation – Scenario 1")
eval_s1 = run_evals(
    plan=initial_plan_s1,
    vacation_info=vacation_info_s1,
    weather_data=weather_s1,
    available_activities=activities_s1,
    client=client,
    model=EVAL_MODEL,
)
print_eval_results(eval_s1)

In [ ]:
revision_s1 = ItineraryRevisionAgent(client=client, model=MAIN_MODEL)

final_plan_s1 = revision_s1.revise(
    plan=initial_plan_s1,
    vacation_info=vacation_info_s1,
    weather_data=weather_s1,
    available_activities=activities_s1,
    eval_model=EVAL_MODEL,
)

elapsed_s1 = time.time() - t0

print("\n📋  Final Itinerary – Scenario 1")
print_itinerary(final_plan_s1)

print("\n🔍  Final Evaluation – Scenario 1")
final_eval_s1 = run_evals(
    plan=final_plan_s1,
    vacation_info=vacation_info_s1,
    weather_data=weather_s1,
    available_activities=activities_s1,
    client=client,
    model=EVAL_MODEL,
)
print_eval_results(final_eval_s1)

print(f"\n📊  Scenario 1 Metrics")
print(f"   Iterations      : {len(revision_s1.reasoning_log)}")
print(f"   Elapsed time    : {elapsed_s1:.1f}s")
print(f"   Total cost      : ${final_plan_s1.total_cost:.2f} / ${vacation_info_s1.budget:.2f}")
print(f"   All checks pass : {final_eval_s1['all_passed']}")
assert final_eval_s1["all_passed"], "❌ Scenario 1 FAILED – not all evaluations passed"
print("\n✅ Scenario 1 PASSED")

---
## Scenario 2 – Adventure Seekers

* **Travelers:** 2 people, outdoor / hiking / sports focus
* **Budget:** $300
* **Duration:** 3 days (2026-06-13 → 2026-06-15)
* **Goal:** Verify weather-compatibility checks and outdoor activity selection

In [ ]:
vacation_info_s2 = VacationInfo(
    destination="AgentsVille",
    start_date="2026-06-13",
    end_date="2026-06-15",
    interests=["outdoor activities", "hiking", "sports", "adventure"],
    budget=300.0,
    constraints=[],
)

weather_s2    = get_weather_forecast(vacation_info_s2)
activities_s2 = get_available_activities(vacation_info_s2, weather_s2)

print("🌤️  Weather Forecast")
for d, w in sorted(weather_s2.items()):
    print(f"   {d}: {w}")

print()
print("🎯  Available Activities per Day")
for d in sorted(activities_s2):
    print(f"   {d} ({weather_s2[d]}): {len(activities_s2[d])} activities")
    for a in activities_s2[d]:
        if a["category"] in ("sports", "outdoor"):
            print(f"      🏃 {a['name']}  –  ${a['cost']:.2f}")

In [ ]:
t0 = time.time()

agent_s2        = ItineraryAgent(client=client, model=MAIN_MODEL)
initial_plan_s2 = agent_s2.generate(
    vacation_info=vacation_info_s2,
    weather_data=weather_s2,
    available_activities=activities_s2,
)

print("📋  Initial Itinerary – Scenario 2")
print_itinerary(initial_plan_s2)

In [ ]:
revision_s2 = ItineraryRevisionAgent(client=client, model=MAIN_MODEL)

final_plan_s2 = revision_s2.revise(
    plan=initial_plan_s2,
    vacation_info=vacation_info_s2,
    weather_data=weather_s2,
    available_activities=activities_s2,
    eval_model=EVAL_MODEL,
)

elapsed_s2 = time.time() - t0

print("\n📋  Final Itinerary – Scenario 2")
print_itinerary(final_plan_s2)

print("\n🔍  Final Evaluation – Scenario 2")
final_eval_s2 = run_evals(
    plan=final_plan_s2,
    vacation_info=vacation_info_s2,
    weather_data=weather_s2,
    available_activities=activities_s2,
    client=client,
    model=EVAL_MODEL,
)
print_eval_results(final_eval_s2)

# Show which days had weather-constrained activity selections
print("\n🏔️  Outdoor Activity Selection")
for day in final_plan_s2.days:
    w = weather_s2[day.date]
    outdoor = [a.name for a in day.activities if any(
        kw in a.name.lower() for kw in ["hik", "kayak", "beach", "boat", "scuba", "dive"]
    )]
    print(f"   {day.date} ({w}): {outdoor or ['no outdoor activities']}")

print(f"\n📊  Scenario 2 Metrics")
print(f"   Iterations      : {len(revision_s2.reasoning_log)}")
print(f"   Elapsed time    : {elapsed_s2:.1f}s")
print(f"   Total cost      : ${final_plan_s2.total_cost:.2f} / ${vacation_info_s2.budget:.2f}")
print(f"   All checks pass : {final_eval_s2['all_passed']}")
assert final_eval_s2["all_passed"], "❌ Scenario 2 FAILED"
print("\n✅ Scenario 2 PASSED")

---
## Scenario 3 – Culture Enthusiast

* **Travelers:** 1 person, art / theatre / museums
* **Budget:** $250
* **Duration:** 4 days (2026-06-10 → 2026-06-13)
* **Goal:** Verify interest matching; inspect ReAct reasoning log

In [ ]:
vacation_info_s3 = VacationInfo(
    destination="AgentsVille",
    start_date="2026-06-10",
    end_date="2026-06-13",
    interests=["art", "museums", "theatre", "culture", "history"],
    budget=250.0,
    constraints=[],
)

weather_s3    = get_weather_forecast(vacation_info_s3)
activities_s3 = get_available_activities(vacation_info_s3, weather_s3)

print("🌤️  Weather Forecast")
for d, w in sorted(weather_s3.items()):
    print(f"   {d}: {w}")

In [ ]:
t0 = time.time()

agent_s3        = ItineraryAgent(client=client, model=MAIN_MODEL)
initial_plan_s3 = agent_s3.generate(
    vacation_info=vacation_info_s3,
    weather_data=weather_s3,
    available_activities=activities_s3,
)

revision_s3 = ItineraryRevisionAgent(client=client, model=MAIN_MODEL)

final_plan_s3 = revision_s3.revise(
    plan=initial_plan_s3,
    vacation_info=vacation_info_s3,
    weather_data=weather_s3,
    available_activities=activities_s3,
    eval_model=EVAL_MODEL,
)

elapsed_s3 = time.time() - t0

print("\n📋  Final Itinerary – Scenario 3")
print_itinerary(final_plan_s3)

print("\n🔍  Final Evaluation – Scenario 3")
final_eval_s3 = run_evals(
    plan=final_plan_s3,
    vacation_info=vacation_info_s3,
    weather_data=weather_s3,
    available_activities=activities_s3,
    client=client,
    model=EVAL_MODEL,
)
print_eval_results(final_eval_s3)

# Show reasoning log
print(f"\n🧠  ReAct Reasoning Log – Scenario 3 ({len(revision_s3.reasoning_log)} entries)")
for i, entry in enumerate(revision_s3.reasoning_log, 1):
    t = entry['type'].upper()
    if t == 'THOUGHT':
        print(f"  [{i}] 💭 THOUGHT: {entry['content'][:200].replace(chr(10), ' ')}")
    elif t == 'ACTION':
        print(f"  [{i}] 🔧 ACTION  → {entry['tool']}")
    elif t == 'FINAL_ANSWER':
        print(f"  [{i}] ✅ FINAL_ANSWER")

print(f"\n📊  Scenario 3 Metrics")
print(f"   Iterations   : {len(revision_s3.reasoning_log)}")
print(f"   Elapsed time : {elapsed_s3:.1f}s")
print(f"   Total cost   : ${final_plan_s3.total_cost:.2f} / ${vacation_info_s3.budget:.2f}")
print(f"   All checks   : {final_eval_s3['all_passed']}")
assert final_eval_s3["all_passed"], "❌ Scenario 3 FAILED"
print("\n✅ Scenario 3 PASSED")

---
## Scenario 4 – Food Lovers

* **Travelers:** 2 people, cooking / food interests
* **Budget:** $400
* **Duration:** 3 days (2026-06-10 → 2026-06-12)
* **Goal:** Verify food-related activities are selected; show cost breakdown

In [ ]:
vacation_info_s4 = VacationInfo(
    destination="AgentsVille",
    start_date="2026-06-10",
    end_date="2026-06-12",
    interests=["food", "cooking", "wine", "local cuisine", "restaurants"],
    budget=400.0,
    constraints=["vegetarian-friendly options preferred"],
)

weather_s4    = get_weather_forecast(vacation_info_s4)
activities_s4 = get_available_activities(vacation_info_s4, weather_s4)

print("🌤️  Weather Forecast")
for d, w in sorted(weather_s4.items()):
    print(f"   {d}: {w}")

In [ ]:
t0 = time.time()

agent_s4        = ItineraryAgent(client=client, model=MAIN_MODEL)
initial_plan_s4 = agent_s4.generate(
    vacation_info=vacation_info_s4,
    weather_data=weather_s4,
    available_activities=activities_s4,
)

revision_s4 = ItineraryRevisionAgent(client=client, model=MAIN_MODEL)

final_plan_s4 = revision_s4.revise(
    plan=initial_plan_s4,
    vacation_info=vacation_info_s4,
    weather_data=weather_s4,
    available_activities=activities_s4,
    eval_model=EVAL_MODEL,
)

elapsed_s4 = time.time() - t0

print("\n📋  Final Itinerary – Scenario 4")
print_itinerary(final_plan_s4)

print("\n🔍  Final Evaluation – Scenario 4")
final_eval_s4 = run_evals(
    plan=final_plan_s4,
    vacation_info=vacation_info_s4,
    weather_data=weather_s4,
    available_activities=activities_s4,
    client=client,
    model=EVAL_MODEL,
)
print_eval_results(final_eval_s4)

# Cost breakdown
print("\n💰  Cost Breakdown – Scenario 4")
for day in final_plan_s4.days:
    print(f"\n   {day.date} – subtotal: ${day.day_total_cost:.2f}")
    for a in day.activities:
        food_tag = "🍽️" if any(kw in a.name.lower() for kw in ["food", "cook", "wine", "market", "taste"]) else "  "
        print(f"   {food_tag}  {a.name}: ${a.cost:.2f}")
print(f"\n   TOTAL: ${final_plan_s4.total_cost:.2f} / budget ${vacation_info_s4.budget:.2f}")

print(f"\n📊  Scenario 4 Metrics")
print(f"   Iterations   : {len(revision_s4.reasoning_log)}")
print(f"   Elapsed time : {elapsed_s4:.1f}s")
print(f"   All checks   : {final_eval_s4['all_passed']}")
assert final_eval_s4["all_passed"], "❌ Scenario 4 FAILED"
print("\n✅ Scenario 4 PASSED")

---
## Scenario 5 – Extended Trip (Scalability Test)

* **Travelers:** 3 people, mixed interests
* **Budget:** $600
* **Duration:** 6 days (2026-06-10 → 2026-06-15)
* **Goal:** Verify the system handles longer itineraries; show performance metrics

In [ ]:
vacation_info_s5 = VacationInfo(
    destination="AgentsVille",
    start_date="2026-06-10",
    end_date="2026-06-15",
    interests=["culture", "food", "outdoor activities", "entertainment", "sports"],
    budget=600.0,
    constraints=["mix of indoor and outdoor activities preferred"],
)

weather_s5    = get_weather_forecast(vacation_info_s5)
activities_s5 = get_available_activities(vacation_info_s5, weather_s5)

print("🌤️  Weather Forecast (6 days)")
for d, w in sorted(weather_s5.items()):
    acts_count = len(activities_s5[d])
    print(f"   {d}: {w}  ({acts_count} activities available)")

In [ ]:
t0 = time.time()

agent_s5        = ItineraryAgent(client=client, model=MAIN_MODEL)
initial_plan_s5 = agent_s5.generate(
    vacation_info=vacation_info_s5,
    weather_data=weather_s5,
    available_activities=activities_s5,
)

revision_s5 = ItineraryRevisionAgent(client=client, model=MAIN_MODEL)

final_plan_s5 = revision_s5.revise(
    plan=initial_plan_s5,
    vacation_info=vacation_info_s5,
    weather_data=weather_s5,
    available_activities=activities_s5,
    eval_model=EVAL_MODEL,
)

elapsed_s5 = time.time() - t0

print("\n📋  Final Itinerary – Scenario 5")
print_itinerary(final_plan_s5)

print("\n🔍  Final Evaluation – Scenario 5")
final_eval_s5 = run_evals(
    plan=final_plan_s5,
    vacation_info=vacation_info_s5,
    weather_data=weather_s5,
    available_activities=activities_s5,
    client=client,
    model=EVAL_MODEL,
)
print_eval_results(final_eval_s5)

print(f"\n📊  Scenario 5 Performance Metrics")
print(f"   Trip days      : {len(final_plan_s5.days)}")
print(f"   Total activities: {sum(len(d.activities) for d in final_plan_s5.days)}")
print(f"   Reasoning steps : {len(revision_s5.reasoning_log)}")
print(f"   Elapsed time   : {elapsed_s5:.1f}s")
print(f"   Total cost     : ${final_plan_s5.total_cost:.2f} / ${vacation_info_s5.budget:.2f}")
print(f"   All checks     : {final_eval_s5['all_passed']}")
assert final_eval_s5["all_passed"], "❌ Scenario 5 FAILED"
print("\n✅ Scenario 5 PASSED")

---
## Scenario 6 – Weather Challenge Test

* **Travelers:** 1 person, outdoor interests
* **Budget:** $200
* **Duration:** 3 days (2026-06-11 → 2026-06-13)
* **Goal:** Some days in this range receive rainy/stormy weather.
  Verify the agent substitutes indoor activities and weather-compatibility passes.

> **Note:** The deterministic weather generator uses `(day * 3 + month * 7) % 8`.
> Days 11–13 of June yield indices that include rainy/stormy conditions.

In [ ]:
vacation_info_s6 = VacationInfo(
    destination="AgentsVille",
    start_date="2026-06-11",
    end_date="2026-06-13",
    interests=["outdoor activities", "hiking", "nature"],
    budget=200.0,
    constraints=["prefer indoor alternatives when weather is poor"],
)

weather_s6    = get_weather_forecast(vacation_info_s6)
activities_s6 = get_available_activities(vacation_info_s6, weather_s6)

print("🌤️  Weather Forecast – Scenario 6")
for d, w in sorted(weather_s6.items()):
    act_names = [a["name"] for a in activities_s6[d]]
    print(f"   {d}: {w}  →  {len(act_names)} activities available")
    for a in activities_s6[d]:
        print(f"      • {a['name']}  (req: {a['weather_requirement']})")

print()
print("ℹ️  Days with restricted weather will have fewer outdoor options.")

In [ ]:
t0 = time.time()

agent_s6        = ItineraryAgent(client=client, model=MAIN_MODEL)
initial_plan_s6 = agent_s6.generate(
    vacation_info=vacation_info_s6,
    weather_data=weather_s6,
    available_activities=activities_s6,
)

revision_s6 = ItineraryRevisionAgent(client=client, model=MAIN_MODEL)

final_plan_s6 = revision_s6.revise(
    plan=initial_plan_s6,
    vacation_info=vacation_info_s6,
    weather_data=weather_s6,
    available_activities=activities_s6,
    eval_model=EVAL_MODEL,
)

elapsed_s6 = time.time() - t0

print("\n📋  Final Itinerary – Scenario 6 (Weather Challenge)")
print_itinerary(final_plan_s6)

print("\n🔍  Final Evaluation – Scenario 6")
final_eval_s6 = run_evals(
    plan=final_plan_s6,
    vacation_info=vacation_info_s6,
    weather_data=weather_s6,
    available_activities=activities_s6,
    client=client,
    model=EVAL_MODEL,
)
print_eval_results(final_eval_s6)

print("\n🌧️  Weather Adaptation Analysis")
for day in final_plan_s6.days:
    w = weather_s6[day.date]
    outdoor_acts = [a.name for a in day.activities
                    if any(kw in a.name.lower() for kw in ["hik", "kayak", "beach", "boat", "scuba"])]
    indoor_acts  = [a.name for a in day.activities if a.name not in outdoor_acts]
    print(f"   {day.date} ({w}):")
    if outdoor_acts:
        print(f"      🏃 Outdoor : {outdoor_acts}")
    if indoor_acts:
        print(f"      🏠 Indoor  : {indoor_acts}")

print(f"\n📊  Scenario 6 Metrics")
print(f"   Iterations   : {len(revision_s6.reasoning_log)}")
print(f"   Elapsed time : {elapsed_s6:.1f}s")
print(f"   Total cost   : ${final_plan_s6.total_cost:.2f} / ${vacation_info_s6.budget:.2f}")
print(f"   All checks   : {final_eval_s6['all_passed']}")
assert final_eval_s6["all_passed"], "❌ Scenario 6 FAILED"
print("\n✅ Scenario 6 PASSED")

---
## 📊 Test Suite Summary

In [ ]:
scenarios = [
    ("Scenario 1 – Budget-Conscious", final_eval_s1, final_plan_s1, vacation_info_s1, elapsed_s1, revision_s1),
    ("Scenario 2 – Adventure Seekers", final_eval_s2, final_plan_s2, vacation_info_s2, elapsed_s2, revision_s2),
    ("Scenario 3 – Culture Enthusiast", final_eval_s3, final_plan_s3, vacation_info_s3, elapsed_s3, revision_s3),
    ("Scenario 4 – Food Lovers",       final_eval_s4, final_plan_s4, vacation_info_s4, elapsed_s4, revision_s4),
    ("Scenario 5 – Extended Trip",     final_eval_s5, final_plan_s5, vacation_info_s5, elapsed_s5, revision_s5),
    ("Scenario 6 – Weather Challenge", final_eval_s6, final_plan_s6, vacation_info_s6, elapsed_s6, revision_s6),
]

print(f"{'Scenario':<35} {'Pass?':<7} {'Cost':>10} {'Budget':>10} {'Steps':>7} {'Time':>7}")
print("-" * 80)
for name, evals, plan, vi, elapsed, rev in scenarios:
    status  = "✅ PASS" if evals["all_passed"] else "❌ FAIL"
    cost    = f"${plan.total_cost:.2f}"
    budget  = f"${vi.budget:.2f}"
    steps   = len(rev.reasoning_log)
    t       = f"{elapsed:.1f}s"
    print(f"{name:<35} {status:<7} {cost:>10} {budget:>10} {steps:>7} {t:>7}")

all_passed = all(e["all_passed"] for _, e, *_ in scenarios)
print()
print(f"Overall result: {'✅ ALL SCENARIOS PASSED' if all_passed else '❌ SOME SCENARIOS FAILED'}  ({sum(1 for _,e,*_ in scenarios if e['all_passed'])}/6)")

---
*Generated by the AgentsVille AI Trip Planner test suite.*